# Joint all-slides pathway analysis

Reproduces `notebooks/application/02_pathway_analysis.ipynb` on the joint 6-slide model
from `01_data_prep.ipynb`.

The single-slide notebook hardcodes `MODULES = ['CRC1', 'CRC2']` with `CRC1 -> TGFb` and
`CRC2 -> NFkB`, but Hotspot's module numbering is arbitrary and the joint run need not find
the same number of modules. So the module -> pathway assignment is **re-derived from PROGENy
activity**: whichever module scores highest for TGFb gets TGFb, and the highest remaining
one for NFkB gets NFkB.

In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import time

sys.path.append('../../scripts')
sys.path.append('../application')

import anndata as ad
import decoupler as dc
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from matplotlib.lines import Line2D

from cellina import Cellina as CellinaModel  # renamed upstream before v0.99.1
from cellina._spatial_utils import make_neighbor_perturbation
from utils import set_seed
from plotting import plot_pathway_activity
from helpers import (build_pw_perturbation, cf_logfc, compute_correlations,
                     compute_microenv_logfc)

import joint
from joint import BATCH_KEY, CELLTYPES, DOMAINS_KEY, LABELS_KEY, build_spatial, slim

In [ ]:
set_seed(0)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 16
plt.rcParams['figure.dpi'] = 100

OUT = 'output'
FIG = '../../figures/application_all_slides'
os.makedirs(FIG, exist_ok=True)

UMAP_MAX_CELLS = 200_000
SELECT_CT = 'Fibroblast'
LOGFC_THRESHOLD = 0.01
WEIGHT_THRESHOLD = 0.5
DEG = 100

## Load 01 outputs

In [ ]:
t0 = time.time()
adata = sc.read_h5ad(f'{OUT}/adata_joint.h5ad')
print(f'loaded {adata.n_obs:,} x {adata.n_vars:,} in {time.time() - t0:.0f}s')

t0 = time.time()
build_spatial(adata)   # rebuild the per-slide graph + spatial_x (not persisted by 01)
print(f'rebuilt spatial features in {time.time() - t0:.0f}s')

model = CellinaModel.load(f'{OUT}/model_final', adata=adata)
print('loaded model')

MODULES = sorted([m for m in adata.obs['microenvironment'].cat.categories if str(m).startswith('CRC')],
                 key=lambda s: int(s[3:]))
print('modules:', MODULES)
display(adata.obs['microenvironment'].value_counts())

## Module x gene matrix

In [ ]:
hs_results = pd.read_csv(f'{OUT}/hotspot_results.csv', index_col=0)
hs_modules = pd.read_csv(f'{OUT}/hotspot_modules.csv', index_col=0)['Module']

df = hs_results[['C']].join(hs_modules)
df = df[~df['Module'].isna() & (df['Module'] != -1.0)]   # drop unassigned genes
module_gene_matrix = df.pivot_table(index='Module', columns=df.index, values='C', fill_value=0)
module_gene_matrix.index = [f'CRC{int(m)}' for m in module_gene_matrix.index]
print(module_gene_matrix.shape)
display(df.groupby('Module').size().rename('n_genes'))
module_gene_matrix

## Pathway activity

In [ ]:
pw_progeny = dc.op.progeny(organism='human')
pw_hallmark = dc.op.hallmark(organism='human')

In [ ]:
pw_acts_progeny, pw_padj_progeny = dc.mt.ulm(data=module_gene_matrix, net=pw_progeny)
plot_pathway_activity(pw_acts_progeny, pw_padj_progeny, alpha=0.05)

In [ ]:
pw_acts_hallmark, pw_padj_hallmark = dc.mt.ulm(data=module_gene_matrix, net=pw_hallmark)
plot_pathway_activity(pw_acts_hallmark, pw_padj_hallmark, alpha=0.05)

In [ ]:
pw_acts_joint = pw_acts_progeny.merge(pw_acts_hallmark, left_index=True, right_index=True).T
pw_padj_joint = pw_padj_progeny.merge(pw_padj_hallmark, left_index=True, right_index=True).T

pw_acts_long = (pw_acts_joint.reset_index()
                .melt(id_vars='index', var_name='Module', value_name='Activity')
                .sort_values(['index', 'Module']))
pw_padj_long = (pw_padj_joint.reset_index()
                .melt(id_vars='index', var_name='Module', value_name='padj')
                .sort_values(['index', 'Module']))
pw_joint = (pw_acts_long.merge(pw_padj_long, on=['index', 'Module'])
            .rename(columns={'index': 'Pathway'}))
pw_joint['Pathway'] = pw_joint['Pathway'].str.replace('EPITHELIAL_MESENCHYMAL_TRANSITION', 'EMT')
pw_joint.to_csv(f'{OUT}/pathway_activity_modules.csv', index=False)
pw_joint.head()

### Module -> pathway assignment

Derived from PROGENy activity only, with no reference to the single-slide run.

In [ ]:
assignment = joint.module_pathway_assignment(pw_acts_progeny, pw_padj_progeny,
                                             pathways=('TGFb', 'NFkB', 'MAPK'))
MOD_TGFB, MOD_NFKB, MOD_MAPK = assignment['TGFb'], assignment['NFkB'], assignment['MAPK']
print('assignment:', assignment)

# The guard (positive activity + padj < 0.05) can legitimately return None -- with min_genes
# large enough to merge the 27 NF-kB genes into the 383-gene EMT module, NFkB activity on
# that module is 1.98 at padj 0.21. The counterfactual/perturbation figures below need *some*
# second module to contrast against, so fall back to the highest-scoring one and carry a
# NFKB_SIG flag into every label, so no figure can be read as a positive NF-kB result.
NFKB_SIG = MOD_NFKB is not None
if not NFKB_SIG:
    MOD_NFKB = pw_acts_progeny['NFkB'].drop(index=[MOD_TGFB]).idxmax()
    print(f'NFkB: NO module passes the guard. Using {MOD_NFKB} for figures only '
          f'(activity {pw_acts_progeny.loc[MOD_NFKB, "NFkB"]:.2f}, '
          f'padj {pw_padj_progeny.loc[MOD_NFKB, "NFkB"]:.3f}) -- not a positive result.')
display(pw_acts_progeny[['TGFb', 'NFkB', 'MAPK']].round(2))
display(pw_padj_progeny[['TGFb', 'NFkB', 'MAPK']].round(4))

### Dotplot over all modules

In [ ]:
PATHWAYS = ['TGFb', 'NFkB', 'MAPK']
CLIP = 5

plt.rcParams.update({'font.family': 'sans-serif', 'font.size': 16})

df = pw_joint[pw_joint['Pathway'].isin(PATHWAYS) & pw_joint['Module'].isin(MODULES)].copy()
df['Activity_clipped'] = df['Activity'].clip(-CLIP, CLIP)
df['neg_log10_padj'] = -np.log10(df['padj'].clip(lower=1e-5))

pathway_order = PATHWAYS[::-1]
module_order = MODULES
df['x'] = df['Module'].map({m: i for i, m in enumerate(module_order)})
df['y'] = df['Pathway'].map({p: i for i, p in enumerate(pathway_order)})

MAX_SIZE, MIN_SIZE = 3000, 200
sig_min, sig_max = df['neg_log10_padj'].min(), df['neg_log10_padj'].max()
size_range = sig_max - sig_min or 1
dot_sizes = MIN_SIZE + (df['neg_log10_padj'] - sig_min) / size_range * (MAX_SIZE - MIN_SIZE)

fig, ax = plt.subplots(figsize=(3 + 2.4 * len(module_order), 6))
for i in range(len(pathway_order)):
    ax.axhspan(i - 0.5, i + 0.5, color='#f5f5f5' if i % 2 == 0 else 'white', zorder=0)
for xi in range(len(module_order)):
    ax.axvline(xi, color='#dddddd', linewidth=0.8, zorder=1)

norm = mcolors.TwoSlopeNorm(vmin=-CLIP, vcenter=0, vmax=CLIP)
xx = ax.scatter(df['x'], df['y'], c=df['Activity_clipped'], s=dot_sizes,
                cmap='RdBu_r', norm=norm, edgecolors='#333333', linewidths=0.5, zorder=3)

PW_OF = {MOD_TGFB: 'TGFb', MOD_NFKB: 'NFkB' if NFKB_SIG else 'NFkB ns', MOD_MAPK: 'MAPK'}
labels = [f'{m}\n({PW_OF[m]})' if m in PW_OF else m for m in module_order]
ax.set_xticks(range(len(module_order)))
ax.set_yticks(range(len(pathway_order)))
ax.set_xticklabels(labels, size=15, fontweight='bold')
ax.set_yticklabels(pathway_order, size=22, fontweight='bold')
ax.xaxis.set_label_position('top')
ax.xaxis.tick_top()
ax.set_xlim(-0.6, len(module_order) - 0.4)
ax.set_ylim(-0.6, len(pathway_order) - 0.4)
ax.tick_params(length=0, pad=8)
sns.despine(ax=ax, left=True, bottom=True, top=True, right=True)

cbar = fig.colorbar(xx, ax=ax, shrink=0.7, pad=0.04, aspect=10, ticks=[-CLIP, 0, CLIP])
cbar.set_label('Activity Score', labelpad=0, rotation=270, va='bottom', fontsize=20)
cbar.ax.tick_params(labelsize=20, length=0)
cbar.outline.set_visible(False)

sig_ticks = np.array([sig_min, sig_min + 0.5 * size_range, sig_max])
size_ticks = MIN_SIZE + (sig_ticks - sig_min) / size_range * (MAX_SIZE - MIN_SIZE)
ax.legend(
    handles=[Line2D([0], [0], marker='o', color='w', markerfacecolor='#888888',
                    markeredgecolor='#333333', markeredgewidth=0.6,
                    markersize=np.sqrt(s) * 0.9, label=f'{v:.1f}')
             for s, v in zip(size_ticks, sig_ticks)],
    title='$-\\log_{10}$(adj. p)', title_fontsize=20, fontsize=20,
    bbox_to_anchor=(1.35, 1.02), loc='upper left', frameon=False,
    labelspacing=0.8, handletextpad=1.2,
)
plt.tight_layout()
fig.savefig(f'{FIG}/all_slides_pathway_dotplot.svg', bbox_inches='tight', dpi=300)
fig.savefig(f'{FIG}/all_slides_pathway_dotplot.png', bbox_inches='tight', dpi=300)
plt.show()
print(f'{FIG}/all_slides_pathway_dotplot.svg')

## Is a module just one patient?

The point of running all slides. A module whose cells come overwhelmingly from one slide is
a patient effect, not a shared programme.

In [ ]:
mod_by_slide = pd.read_csv(f'{OUT}/module_slide_composition.csv', index_col=0)
mod_frac = mod_by_slide.div(mod_by_slide.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(1.6 * len(mod_frac) + 3, 4.5))
mod_frac.loc[MODULES].plot(kind='bar', stacked=True, ax=ax,
                           colormap='tab20', width=0.75, edgecolor='white', linewidth=0.5)
ax.set_ylabel('fraction of module cells')
ax.set_xlabel('')
ax.set_ylim(0, 1)
ax.axhline(1 / mod_frac.shape[1], color='0.3', ls='--', lw=1)
PW_OF = {MOD_TGFB: 'TGFb', MOD_NFKB: 'NFkB' if NFKB_SIG else 'NFkB ns', MOD_MAPK: 'MAPK'}
ax.set_xticklabels([f'{m}\n({PW_OF[m]})' if m in PW_OF else m for m in MODULES], rotation=0)
ax.legend(title='slide', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
sns.despine(ax=ax)
plt.tight_layout()
fig.savefig(f'{FIG}/all_slides_module_composition.svg', bbox_inches='tight', dpi=300)
fig.savefig(f'{FIG}/all_slides_module_composition.png', bbox_inches='tight', dpi=300)
plt.show()

print('max single-slide share per module (%):')
display((mod_frac.loc[MODULES].max(axis=1) * 100).round(1))

## Counterfactual fibroblast UMAP

In [ ]:
fib = sc.read_h5ad(f'{OUT}/cf_{SELECT_CT}.h5ad')
n_ctrl = int(fib.uns['n_control'])
ctrl, real = fib[:n_ctrl], fib[n_ctrl:]
print(f'{n_ctrl:,} control + {real.n_obs:,} observed CRC-region {SELECT_CT}s')
display(ctrl.obs[DOMAINS_KEY].value_counts())
display(real.obs['microenvironment'].value_counts())

In [ ]:
SHOW_MODULES = [MOD_TGFB, MOD_NFKB]

arms = []
is_ref = (ctrl.obs[DOMAINS_KEY].astype(str) == 'REF').to_numpy()
obs = ctrl.obs[is_ref].copy()
obs['domain'] = 'Control'
arms.append(ad.AnnData(X=np.log1p(ctrl.obsm['recon_x'][is_ref]), obs=obs))

for m in SHOW_MODULES:
    in_m = (real.obs['microenvironment'].astype(str) == m).to_numpy()
    obs = real.obs[in_m].copy()
    obs['domain'] = m
    arms.append(ad.AnnData(X=np.log1p(real.obsm['recon_x'][in_m]), obs=obs))

    obs = ctrl.obs.copy()
    obs['domain'] = f'{m} CF'
    arms.append(ad.AnnData(X=np.log1p(fib.uns[f'counterfactual_x_{m}']), obs=obs))

merged = ad.concat(arms, join='outer')
merged.obs_names_make_unique()
merged.var_names = fib.var_names
print(merged.obs['domain'].value_counts())

if merged.n_obs > UMAP_MAX_CELLS:
    rng = np.random.default_rng(0)
    merged = merged[np.sort(rng.choice(merged.n_obs, UMAP_MAX_CELLS, replace=False))].copy()
    print(f'subsampled to {merged.n_obs:,} for the UMAP')

In [ ]:
# Centre each slide's cells per gene before embedding. cellina's
# get_normalized_expression / get_counterfactual_expression take no transform_batch, so every
# cell is decoded with its OWN batch covariate and all five arms carry their slide effect into
# the plot. Measured on the uncentred version, the embedding was 85% slide (eta^2 0.850) and
# 1% arm (0.011) -- i.e. a batch plot. Centring drops slide to 0.006 and raises the arm
# contrast to 0.141. Expect a diffuse cloud with gradients, not discrete clusters; the crisp
# blobs in the uncentred version WERE the slides. See _umap_diag.py.
_X = merged.X if isinstance(merged.X, np.ndarray) else merged.X.toarray()
_sid = merged.obs['sid'].astype(str).to_numpy()
for _s in np.unique(_sid):
    _m = _sid == _s
    _X[_m] -= _X[_m].mean(0)
merged.X = _X
del _X

sc.pp.pca(merged, n_comps=50)
sc.pp.neighbors(merged, n_pcs=20)
sc.tl.umap(merged)

In [ ]:
base = {MOD_TGFB: '#580803', MOD_NFKB: '#011080'}
light = {MOD_TGFB: '#F4A582', MOD_NFKB: '#92C5DE'}
palette = {'Control': '#BBBBBB'}
for m in SHOW_MODULES:
    palette[m] = base[m]
    palette[f'{m} CF'] = light[m]

plot_order = [d for m in SHOW_MODULES for d in (m, f'{m} CF')]
sub = merged[merged.obs['domain'].isin(plot_order)].copy()
sub.obs['domain'] = pd.Categorical(sub.obs['domain'], categories=plot_order)

ax = sc.pl.umap(sub, color='domain', palette=palette, show=False, title='',
                size=8, alpha=0.7, frameon=False)
fig = ax.figure
fig.set_size_inches(6, 4)
ax.set_position([0.05, 0.08, 0.72, 0.84])
ax.set_title(f'{SELECT_CT}: Control \u2192 Target', fontsize=26, pad=12, loc='left')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])

arrow_kw = dict(arrowstyle='->', color='0.3', lw=1.6)
x0, y0 = ax.get_xlim()[0], ax.get_ylim()[0]
offset = (ax.get_xlim()[1] - x0) * 0.08
ax.annotate('', xy=(x0 + offset * 2, y0), xytext=(x0, y0), arrowprops=arrow_kw, annotation_clip=False)
ax.annotate('', xy=(x0, y0 + offset * 2), xytext=(x0, y0), arrowprops=arrow_kw, annotation_clip=False)
ax.text(x0 + offset * 2.2, y0, 'UMAP1', fontsize=26, color='0.3', va='center')
ax.text(x0, y0 + offset * 2.2, 'UMAP2', fontsize=26, color='0.3', ha='center')

legend = ax.get_legend()
if legend is not None:
    legend.set_title('')
    legend.get_frame().set_linewidth(0)
    legend.get_frame().set_alpha(0)
    for text in legend.get_texts():
        text.set_fontsize(22)
    for handle in legend.legend_handles:
        handle.set_sizes([80])
    legend.set_bbox_to_anchor((1.02, 1))
    legend.set_loc('upper left')

for coll in ax.collections:
    coll.set_rasterized(True)

fig.savefig(f'{FIG}/all_slides_umap_cf_{SELECT_CT}.svg', dpi=300, bbox_inches='tight', transparent=True)
fig.savefig(f'{FIG}/all_slides_umap_cf_{SELECT_CT}.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'{FIG}/all_slides_umap_cf_{SELECT_CT}.svg')

## Pathway-based perturbations

`Control` is REF across all slides (`typ_clean == 'REF'`), matching the single-slide
notebook's REF-only reference.

In [ ]:
micro = adata.obs['microenvironment'].astype(str).replace({'REF': 'Control'})
adata.obs['microenv_ctrl'] = pd.Categorical(micro)
display(adata.obs['microenv_ctrl'].value_counts())

In [ ]:
logfc = {}
for m in (MOD_TGFB, MOD_NFKB):
    keep = np.flatnonzero(adata.obs['microenv_ctrl'].astype(str).isin(['Control', m]).to_numpy())
    sub = slim(adata, keep, obs_cols=['microenv_ctrl', LABELS_KEY, 'sid'])
    g, ct_df = compute_microenv_logfc(sub, domains_key='microenv_ctrl', labels_key=LABELS_KEY,
                                      ref_label='Control', crc_label=m)
    logfc[m] = g
    print(f'{m}: pseudobulk logFC over {sub.n_obs:,} cells')
    del sub

In [ ]:
pw_tgfb = pw_progeny[(pw_progeny.source == 'TGFb') &
                     (pw_progeny.weight.abs() > WEIGHT_THRESHOLD)].copy()
pw_nfkb = pw_progeny[(pw_progeny.source == 'NFkB') &
                     (pw_progeny.weight.abs() > WEIGHT_THRESHOLD)].copy()
pw_tgfb['weight'] = pw_tgfb['weight'].clip(-5, 5)
pw_nfkb['weight'] = pw_nfkb['weight'].clip(-5, 5)
print(f'TGFb targets {len(pw_tgfb)}, NFkB targets {len(pw_nfkb)}')

pert = {
    MOD_TGFB: build_pw_perturbation(logfc[MOD_TGFB], [pw_tgfb], LOGFC_THRESHOLD),
    MOD_NFKB: build_pw_perturbation(logfc[MOD_NFKB], [pw_nfkb], LOGFC_THRESHOLD),
}
for m, p in pert.items():
    print(f'{m}: {len(p)} perturbation genes')
    assert len(p) > 0, f'no perturbation genes for {m}'

In [ ]:
# control fibroblasts, the same cells 01 used for recon_x / counterfactuals
idx_ctrl = adata.obs_names.get_indexer(ctrl.obs_names)
assert (idx_ctrl >= 0).all()
print(f'{len(idx_ctrl):,} control {SELECT_CT}s')

recon_ctrl = ctrl.obsm['recon_x']

# make_neighbor_perturbation reads adata.X and must see the same log-normalised space
# that spatial_x was built from (build_spatial leaves counts in X).
adata.X = adata.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

perturbed = {}
for m, p in pert.items():
    t0 = time.time()
    make_neighbor_perturbation(adata, perturbations=p, groupby=None,
                               obsm_key_out='spatial_x_cf', base=np.e)
    perturbed[m] = model.get_perturbed_expression(
        adata, indices=idx_ctrl, batch_size=4096,
        spatial_obsm_key='spatial_x_cf', library_size=1e4)
    print(f'{m}: perturbed expression in {(time.time() - t0) / 60:.1f} min')
    del adata.obsm['spatial_x_cf']
    if 'counts_cf' in adata.layers:
        del adata.layers['counts_cf']

adata.X = adata.layers['counts'].copy()

### Validate against the observed microenvironments

In [ ]:
fib_ln = fib.copy()
fib_ln.X = fib_ln.layers['counts'].copy()
sc.pp.normalize_total(fib_ln, target_sum=1e4)
sc.pp.log1p(fib_ln)
Xln = np.asarray(fib_ln.X.todense())

control_expr = Xln[:n_ctrl]
pert_results = {}
for m in (MOD_TGFB, MOD_NFKB):
    in_m = np.flatnonzero((fib.obs['microenvironment'].astype(str) == m).to_numpy())
    target_expr = Xln[in_m]
    pear, spear = compute_correlations(control_expr, target_expr,
                                       np.log1p(perturbed[m]), deg=DEG)
    pw = 'TGFb' if m == MOD_TGFB else 'NFkB'
    pert_results[f'{SELECT_CT} CF ({m}, {pw})'] = {'pearson': pear, 'spearman': spear,
                                                   'n_target': len(in_m)}
    print(f'{m} ({pw}) - Pearson {pear:.3f}, Spearman {spear:.3f}  (n_target={len(in_m):,})')

pd.DataFrame(pert_results).T.to_csv(f'{OUT}/perturbation_correlations.csv')
display(pd.DataFrame(pert_results).T)

### Counterfactual logFC scatter

In [ ]:
lfc = {m: cf_logfc(perturbed[m], recon_ctrl) for m in (MOD_TGFB, MOD_NFKB)}
for m, v in lfc.items():
    print(f'{m}: {(np.abs(v) > 0.5).sum()} genes with |logFC|>0.5')

In [ ]:
from adjustText import adjust_text

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'svg.fonttype': 'none',
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
})

top_k = 50
n_label = 3
FS = 24
gene_names = np.array(adata.var_names)
lfc_a, lfc_b = lfc[MOD_TGFB], lfc[MOD_NFKB]
col_a, col_b, col_shared = '#580803', '#011080', 'mediumpurple'

in_top_a = np.isin(gene_names, gene_names[np.argsort(-np.abs(lfc_a))[:top_k]])
in_top_b = np.isin(gene_names, gene_names[np.argsort(-np.abs(lfc_b))[:top_k]])
in_both = in_top_a & in_top_b

fig, ax = plt.subplots(figsize=(9, 5))
for mask, col, lbl, z, alpha, s in [
    (~in_top_a & ~in_top_b, '#d0d0d0', 'Other', 0, 0.3, 20),
    (in_top_a & ~in_top_b, col_a, 'TGFb', 1, 0.6, 30),
    (~in_top_a & in_top_b, col_b, 'NFkB', 1, 0.6, 30),
    (in_both, col_shared, 'Shared', 2, 0.9, 30),
]:
    ax.scatter(lfc_a[mask], lfc_b[mask], c=col, s=s, alpha=alpha, label=lbl,
               zorder=z, linewidths=0, rasterized=True)

ax.axhline(0, color='#aaaaaa', lw=0.6, ls='--', zorder=0)
ax.axvline(0, color='#aaaaaa', lw=0.6, ls='--', zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

texts = []
def label_top(mask, score, color):
    idx = np.where(mask)[0]
    for i in idx[np.argsort(-np.abs(score[idx]))[:n_label]]:
        texts.append(ax.text(lfc_a[i], lfc_b[i], gene_names[i],
                             fontsize=FS - 1, color=color, fontweight='bold', zorder=5))

label_top(in_top_a & ~in_top_b, lfc_a, col_a)
label_top(~in_top_a & in_top_b, lfc_b, col_b)
adjust_text(texts, ax=ax, expand=(1.3, 1.6),
            arrowprops=dict(arrowstyle='-', color='#888888', lw=0.7, shrinkA=1, shrinkB=2),
            force_text=(0.6, 0.9), force_points=(0.4, 0.6))

ax.set_xlabel(f'logFC  TGFb / {MOD_TGFB}', fontsize=FS - 4)
ax.set_ylabel(f'logFC  NFkB{"" if NFKB_SIG else " (ns)"} / {MOD_NFKB}', fontsize=FS - 4)
ax.tick_params(labelsize=FS - 3)
ax.legend(markerscale=2, fontsize=FS - 2, loc='center left', bbox_to_anchor=(1.03, 0.5),
          borderaxespad=0, frameon=False, handletextpad=0.4, labelspacing=0.5)

plt.tight_layout()
fig.savefig(f'{FIG}/all_slides_cf_lfc_scatter.svg', format='svg', bbox_inches='tight', dpi=300)
fig.savefig(f'{FIG}/all_slides_cf_lfc_scatter.png', bbox_inches='tight', dpi=300)
plt.show()
print(f'{FIG}/all_slides_cf_lfc_scatter.svg')

## Summary

In [ ]:
print('slides      :', sorted(adata.obs['sid'].astype(str).unique()))
print('patients    :', adata.obs['pid'].nunique())
print('cells       :', f'{adata.n_obs:,}')
print('modules     :', MODULES)
print('assignment  :', assignment)
print('max single-slide share per module (%):')
print((mod_frac.loc[MODULES].max(axis=1) * 100).round(1).to_string())
print('\nperturbation validation:')
print(pd.DataFrame(pert_results).T.to_string())